In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO

import os
import glob
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF


#images_dir = glob.glob(os.path.join("/kaggle/input/q3-stage3-2026/dataset/images"))
#masks_dir  = glob.glob(os.path.join("/kaggle/input/q3-stage3-2026/dataset/masks"))

# i face problem with this data first i use glob to collect all path and it give me error told me there is .jpeg not found so i make recursive function i try it a lot until it works and i am not sure if it is correct
def img_p(root, keywords):
    candidates = []
    for d in glob.glob(os.path.join(root, "**"), recursive=True):
        if os.path.isdir(d): # to check if class thier or not
            name = os.path.basename(d).lower()
            if any(k in name for k in keywords):
                imgs = glob.glob(os.path.join(d, "/*.*"))  # here i collect all type of extension in folder
                if len(imgs) > 0:
                    candidates.append(d)
    return candidates[0] if len(candidates) > 0 else None

images_dir = img_p(path, ["img", "image", "images", "rgb", "input"])
masks_dir  = img_p(path, ["mask", "masks", "label", "labels", "gt", "seg", "annotation", "ann"])

print("Images dir:", images_dir)
print("Masks dir:", masks_dir)

# create dataset
class SUIMDataset(Dataset):
    def __init__(self, images_dir, masks_dir, img_size=(256, 256)):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.img_size = img_size

        self.image_paths = sorted(
            glob.glob(os.path.join(images_dir, "*.jpg")) +
            glob.glob(os.path.join(images_dir, "*.png")) +
            glob.glob(os.path.join(images_dir, "*.jpeg"))
        )

        # Build a dict for masks
        mask_paths = glob.glob(os.path.join(masks_dir, "*.png")) + glob.glob(os.path.join(masks_dir, "*.jpg")) + glob.glob(os.path.join(masks_dir, "*.jpeg"))
        self.mask_map = {os.path.splitext(os.path.basename(p))[0]: p for p in mask_paths}

        # just to filter if thier or not
        paired = []
        for ip in self.image_paths:
            stem = os.path.splitext(os.path.basename(ip))[0]
            if stem in self.mask_map:
                paired.append(ip)
        self.image_paths = paired

        # Transform
        self.img_tf = transforms.Compose([
            transforms.Resize(self.img_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):   # the sizee
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        stem = os.path.splitext(os.path.basename(img_path))[0]
        mask_path = self.mask_map[stem]

        image = Image.open(img_path).convert("RGB")
        mask  = Image.open(mask_path)

        # first he told there error in size try to resize i take image and mask and i resize it and its work
        image = TF.resize(image, self.img_size)
        mask  = TF.resize(mask,  self.img_size)

        image = self.img_tf(image)

        # mask -> LongTensor [H, W]
        mask = torch.from_numpy(np.array(mask)).long()

        # Optional remap to consecutive labels starting at 0
        mask = remap_mask(mask)

        return image, mask


# Split train/val i split manualy i try a lot in diffrent way and i try with it manually and alhmdulleh its work
dataset = SUIMDataset(images_dir, masks_dir, img_size=(256, 256))
n = len(dataset)
val_size = int(0.2 * n)
train_size = n - val_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

print("Total:", n, "Train:", len(train_dataset), "Val:", len(val_dataset))




In [ ]:
# i display 3 images with thier mask
imgs, msks = next(iter(train_loader))

fig, axes = plt.subplots(3, 2, figsize=(10, 10))
for i in range(3):
    # denormalize for display
    img = imgs[i].cpu().permute(1, 2, 0).numpy()
    img = (img * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
    img = np.clip(img, 0, 1)

    axes[i, 0].imshow(img)
    axes[i, 0].set_title("Image")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(msks[i].cpu().numpy(), vmin=0, vmax=7)
    axes[i, 1].set_title("Mask")
    axes[i, 1].axis("off")



plt.tight_layout()
plt.show()

In [ ]:
!pip install segmentation-models-pytorch


In [ ]:
# TO DO

import torch
import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
## i define the model and whigt
model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=8,
    activation=None
).to(device)

print(device)
print(model)


In [ ]:
# TO DO

from tqdm import tqdm # for bar progras
import torch.nn.functional as F
# Training loop
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for images, masks in tqdm(loader):
        images = images.to(device)
        masks  = masks.to(device)  ## all to device

        optimizer.zero_grad()
        outputs = model(images)   ## all to device
        loss = criterion(outputs, masks) # compute loss

        loss.backward()  ## backpropagation
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

# 🔹 Validation Loop

def validate(model, loader, criterion, device):
    model.eval()  ## ecal mode
    total_loss = 0.0

    with torch.no_grad(): # Disable gradient computation
        for images, masks in loader:
            images = images.to(device)
            masks  = masks.to(device)

            outputs = model(images) # Forward pass
            loss = criterion(outputs, masks)  # compute loss

            total_loss += loss.item()

    return total_loss / len(loader)


In [ ]:
# TO DO

import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

#  define Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 1 ## it takes so longg no much time left :((

train_losses = []
val_losses = []

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss   = validate(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

# Plot curve for loss
plt.figure(figsize=(6,4))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()
plt.show()


In [ ]:
# TO DO

import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F

model.eval()
images, masks = next(iter(val_loader))
images = images.to(device)

with torch.no_grad():## make grad disable
    outputs = model(images)                 ## all to device
    preds = torch.argmax(outputs, dim=1)    ## all to device

images_cpu = images.cpu()
masks_cpu = masks.cpu()
preds_cpu = preds.cpu()

fig, axes = plt.subplots(4, 3, figsize=(10, 12))
# plottt it
for i in range(4):
    img = images_cpu[i].permute(1,2,0).numpy()
    img = (img * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
    img = np.clip(img, 0, 1)

    axes[i, 0].imshow(img)
    axes[i, 0].set_title("Image")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(masks_cpu[i].numpy(), vmin=0, vmax=7)
    axes[i, 1].set_title("Ground Truth")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(preds_cpu[i].numpy(), vmin=0, vmax=7)
    axes[i, 2].set_title("Prediction")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()
